# 01 — Natural Recovery Propensity

METIS buildathon ML training notebook. Uses the same training logic as `train_metis_models.py`.

Natural recovery model: train only on CONTROL observations so the model estimates `P(payment | no intervention)`.

In [ ]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, log_loss, roc_auc_score
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
BASE_FEATURES = [
    "amount", "days_overdue", "previous_failed_count",
    "historical_payment_rate", "contact_count",
    "previous_no_response_count", "segment_high_value",
    "segment_at_risk", "failure_insufficient_funds",
]
INTERVENTIONS = ["RETRY", "REMINDER", "PAYMENT_LINK", "PAYMENT_PLAN", "NEGOTIATION"]

ROOT = Path.cwd()
for candidate in [
    ROOT / "recovery_events.csv",
    ROOT / "../recovery_events.csv",
    ROOT / "../data/recovery_events.csv",
    ROOT / "../../ml_data/recovery_events.csv",
]:
    if candidate.exists():
        DATA_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError("Place recovery_events.csv in the notebook directory, ../, ../data/, or ../../ml_data/")

OUTPUT_DIR = ROOT / "../backend/models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)


In [ ]:

def feature_frame(df):
    out = pd.DataFrame(index=df.index)
    for name in [
        "amount", "days_overdue", "previous_failed_count",
        "historical_payment_rate", "contact_count",
        "previous_no_response_count"
    ]:
        out[name] = pd.to_numeric(df[name], errors="coerce").fillna(0.0)

    out["segment_high_value"] = (
        df["segment"].fillna("REGULAR").astype(str).str.upper() == "HIGH_VALUE"
    ).astype(float)
    out["segment_at_risk"] = (
        df["segment"].fillna("REGULAR").astype(str).str.upper() == "AT_RISK"
    ).astype(float)
    out["failure_insufficient_funds"] = (
        df["failure_reason"].fillna("").astype(str).str.lower().str.contains("insufficient")
    ).astype(float)
    return out[BASE_FEATURES].astype(float)

def make_classifier(seed=RANDOM_STATE):
    return LGBMClassifier(
        objective="binary",
        n_estimators=260,
        learning_rate=0.035,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=40,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.60,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )


In [ ]:

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["intervention_type"]
)
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

control_train = train_df[train_df["intervention_type"].astype(str).str.upper() == "CONTROL"]
control_test = test_df[test_df["intervention_type"].astype(str).str.upper() == "CONTROL"]

model = make_classifier()
model.fit(
    feature_frame(control_train),
    control_train["paid_after_intervention"].astype(int)
)


In [ ]:

proba = model.predict_proba(feature_frame(control_test))[:, 1]
pred = (proba >= 0.5).astype(int)

metrics = {
    "auc_pr": float(average_precision_score(control_test["paid_after_intervention"], proba)),
    "roc_auc": float(roc_auc_score(control_test["paid_after_intervention"], proba)),
    "brier": float(brier_score_loss(control_test["paid_after_intervention"], proba)),
    "log_loss": float(log_loss(control_test["paid_after_intervention"], proba, labels=[0, 1])),
    "accuracy_at_0_5": float(accuracy_score(control_test["paid_after_intervention"], pred)),
}
metrics


In [ ]:

artifact = {
    "model": model,
    "feature_names": BASE_FEATURES,
    "model_version": "metis-propensity-lgbm-v1",
    "training_policy": "CONTROL_ONLY",
}
joblib.dump(artifact, OUTPUT_DIR / "propensity_model.pkl", compress=3)

with open(OUTPUT_DIR / "propensity_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", OUTPUT_DIR / "propensity_model.pkl")
print(metrics)
